<a href="https://colab.research.google.com/github/bilgeerol/bim448-sentiment/blob/main/notebooks/01_veri_hazirlama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 0. KURULUM HÜCRESİ — her oturumda ilk bu çalıştırılır
from google.colab import userdata, drive
import os

drive.mount('/content/drive')

TOKEN = userdata.get('GITHUB_TOKEN')
REPO_OWNER = "bilgeerol"
REPO = "bim448-sentiment"
REPO_DIR = f"/content/{REPO}"

if not os.path.exists(REPO_DIR):
    !git clone https://{TOKEN}@github.com/{REPO_OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git config user.name "zehranurBurmaoglu"
!git config user.email "znburmaoglu@gmail.com"

# Kontrol: depo geldi mi, push iznim var mı?
!ls
!git push --dry-run

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/bim448-sentiment'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 17 (delta 3), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 17.26 KiB | 4.31 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/bim448-sentiment
Already up to date.
data  notebooks  predictions  README.md  report  requirements.txt  results  src
Everything up-to-date


# 01 – Veri Hazırlama
**Sahibi:** Zehra · **Amaç:** İngilizce (Amazon Reviews 2023, All_Beauty) ve Türkçe (TRSAv1) yorumları indirmek, üç sınıfa etiketlemek, filtrelemek ve train/val/test olarak bölmek.
**Girdi:** Hugging Face'teki TRSAv1.csv, Drive'daki `bim448/raw/raw/review_categories/All_Beauty.jsonl`
**Çıktı:** `data/processed/` altında `{en,tr}_{train,val,test}.csv`

## 1. Veri setlerini indir ve incele

In [ ]:
# 1.1 TRSAv1'i doğrudan Hugging Face'ten oku ve genel yapısını yazdır
!pip install -q datasets huggingface_hub
import pandas as pd

TR_URL = "hf://datasets/maydogan/Turkish_SentimentAnalysis_TRSAv1/TRSAv1.csv"
df_tr = pd.read_csv(TR_URL)

print("Boyut (satır, sütun):", df_tr.shape)
print("Sütunlar:", list(df_tr.columns))
print("\nEksik değer sayıları:\n", df_tr.isna().sum())
display(df_tr.head(10))

# Az sayıda farklı değeri olan sütunlar büyük ihtimalle etiket sütunudur
for sutun in df_tr.columns:
    if df_tr[sutun].nunique() <= 10:
        print(f"\n'{sutun}' sütununun değer dağılımı:")
        print(df_tr[sutun].value_counts(dropna=False))

Boyut (satır, sütun): (150000, 3)
Sütunlar: ['id', 'score', 'review']

Eksik değer sayıları:
 id        0
score     0
review    0
dtype: int64


,id,score,review
0,0,Positive,Kokusu çok güzel etkisi de efsane
1,1,Negative,hiç bir fark olmadı siyah nokta silmedi
2,2,Positive,şeffaf güzel bir krem elimi yumuşacık yaptı
3,3,Positive,Özellikle karma ve hassas cildiniz varsa La Ro...
4,4,Negative,çok iyi nemlendiriyor ama daha ucuz muadilleri...
5,5,Neutral,Ürün fena değil güzel tabi öbür kimyasallar gi...
6,6,Negative,şampuan çok köttü
7,7,Negative,Ürün kullanıp memnun olduğum bir ürün fakat şi...
8,8,Neutral,yeni kullaniyorum dokulmeleri azaltır inşallah
9,9,Positive,Kesinlikle saçlarınızda sarı işlem barsa hafta...



'score' sütununun değer dağılımı:
score
Positive    50000
Negative    50000
Neutral     50000
Name: count, dtype: int64


In [ ]:
# 1.2 All_Beauty yorumlarını McAuley Lab'in Hugging Face deposundan Drive'a indir ve ilk kaydı incele
import os, json
from huggingface_hub import hf_hub_download

RAW_DIR = "/content/drive/MyDrive/bim448/raw"

EN_FILE = hf_hub_download(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    repo_type="dataset",
    filename="raw/review_categories/All_Beauty.jsonl",
    local_dir=RAW_DIR,
)
print("İndirilen dosya:", EN_FILE)
print("Dosya boyutu (MB):", round(os.path.getsize(EN_FILE) / 1024**2, 1))

# Yalnızca ilk satırı oku (tamamını belleğe almadan)
with open(EN_FILE, "r", encoding="utf-8") as f:
    ilk_kayit = json.loads(next(f))

print("\nBir kayıttaki alanlar:", list(ilk_kayit.keys()))
print("rating:", ilk_kayit["rating"])
print("text  :", ilk_kayit["text"][:300])

raw/review_categories/All_Beauty.jsonl: reconstructing file:   0%|          |  0.00B /  327MB            

raw/review_categories/All_Beauty.jsonl: downloading bytes:           |  0.00B            

İndirilen dosya: /content/drive/MyDrive/bim448/raw/raw/review_categories/All_Beauty.jsonl
Dosya boyutu (MB): 311.5

Bir kayıttaki alanlar: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
rating: 5.0
text  : This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!


In [ ]:
# 1.3 All_Beauty dosyasını satır satır oku, yalnızca rating ve text alanlarını tut, dağılımı incele
kayitlar = []
with open(EN_FILE, "r", encoding="utf-8") as f:
    for satir in f:
        kayit = json.loads(satir)
        kayitlar.append({"rating": kayit.get("rating"), "text": kayit.get("text")})

df_en = pd.DataFrame(kayitlar)
del kayitlar  # ara listeyi bellekten sil

print("Boyut (satır, sütun):", df_en.shape)
print("\nEksik değer sayıları:\n", df_en.isna().sum())
print("\nBoş (yalnızca boşluk) metin sayısı:", (df_en["text"].fillna("").str.strip() == "").sum())

print("\n'rating' dağılımı:")
print(df_en["rating"].value_counts().sort_index())

# Ön bakış: yıldızlar üç sınıfa çevrilince kaç yorum düşüyor? (filtre öncesi, kesin sayılar Z2'de)
on_bakis = df_en["rating"].map({1.0: "negative", 2.0: "negative", 3.0: "neutral",
                                4.0: "positive", 5.0: "positive"})
print("\nFiltre öncesi üç sınıf dağılımı:")
print(on_bakis.value_counts())

display(df_en.head(5))

Boyut (satır, sütun): (701528, 2)

Eksik değer sayıları:
 rating    0
text      0
dtype: int64

Boş (yalnızca boşluk) metin sayısı: 720

'rating' dağılımı:
rating
1.0    102080
2.0     43034
3.0     56307
4.0     79381
5.0    420726
Name: count, dtype: int64

Filtre öncesi üç sınıf dağılımı:
rating
positive    500107
negative    145114
neutral      56307
Name: count, dtype: int64


,rating,text
0,5.0,This spray is really nice. It smells really go...
1,4.0,"This product does what I need it to do, I just..."
2,5.0,"Smells good, feels great!"
3,1.0,Felt synthetic
4,5.0,Love it


## 2. Etiketle, filtrele, böl, kaydet

In [ ]:
# 2.1 Etiketleri ortak biçime (negative/neutral/positive) çevir, sütunları text ve label olarak adlandır
SINIFLAR = ["negative", "neutral", "positive"]

EN_ETIKET = {1.0: "negative", 2.0: "negative", 3.0: "neutral", 4.0: "positive", 5.0: "positive"}
en = pd.DataFrame({"text": df_en["text"], "label": df_en["rating"].map(EN_ETIKET)})

TR_ETIKET = {"Negative": "negative", "Neutral": "neutral", "Positive": "positive"}
tr = pd.DataFrame({"text": df_tr["review"], "label": df_tr["score"].map(TR_ETIKET)})

for ad, df in [("EN", en), ("TR", tr)]:
    print(f"{ad}: {len(df)} satır | eşleşmeyen etiket sayısı: {df['label'].isna().sum()}")
    print(df["label"].value_counts().reindex(SINIFLAR), "\n")

EN: 701528 satır | eşleşmeyen etiket sayısı: 0
label
negative    145114
neutral      56307
positive    500107
Name: count, dtype: int64 

TR: 150000 satır | eşleşmeyen etiket sayısı: 0
label
negative    50000
neutral     50000
positive    50000
Name: count, dtype: int64 



In [ ]:
# 2.2 Temizlik: iki dilde aynı sırayla uygula, her adımdan sonra satır sayısını kaydet
import re

def temizle(df, dil):
    df = df.copy()
    sayac = [("Başlangıç", len(df))]

    # Metni düzelten adımlar (satır silmez)
    if dil == "EN":
        df["text"] = df["text"].str.replace(r"<br\s*/?>", " ", regex=True, flags=re.IGNORECASE)
    df["text"] = df["text"].str.replace(r"\s+", " ", regex=True).str.strip()

    # Boş ve NaN metinleri sil
    df = df[df["text"].notna() & (df["text"] != "")]
    sayac.append(("Boş metinler silindikten sonra", len(df)))

    # 3–150 kelime arası yorumları tut
    kelime = df["text"].str.split().str.len()
    df = df[(kelime >= 3) & (kelime <= 150)]
    sayac.append(("3–150 kelime filtresinden sonra", len(df)))

    # Birebir aynı metinlerin yalnızca ilkini bırak
    celiskili = (df.groupby("text")["label"].nunique() > 1).sum()
    df = df.drop_duplicates(subset="text")
    sayac.append(("Tekrarlar silindikten sonra", len(df)))

    print(f"{dil}: farklı etiketlerle tekrarlanan metin sayısı: {celiskili}")
    return df.reset_index(drop=True), sayac

en_temiz, en_sayac = temizle(en, "EN")
tr_temiz, tr_sayac = temizle(tr, "TR")

adim_tablosu = pd.DataFrame({"Adım": [a for a, _ in en_sayac],
                             "EN": [n for _, n in en_sayac],
                             "TR": [n for _, n in tr_sayac]})
display(adim_tablosu)

# 2.3 Kontrol: her sınıfta en az 12.000 yorum olmalı (1.000 test + 1.000 val + 10.000 train)
for ad, df in [("EN", en_temiz), ("TR", tr_temiz)]:
    sayim = df["label"].value_counts().reindex(SINIFLAR)
    print(f"\n{ad} sınıf dağılımı (temizlik sonrası):")
    for sinif, n in sayim.items():
        print(f"  {sinif:9s} {n:7d}  {'✓' if n >= 12000 else '✗ 12.000 altında!'}")

EN: farklı etiketlerle tekrarlanan metin sayısı: 418
TR: farklı etiketlerle tekrarlanan metin sayısı: 612


,Adım,EN,TR
0,Başlangıç,701528,150000
1,Boş metinler silindikten sonra,700808,150000
2,3–150 kelime filtresinden sonra,635097,142063
3,Tekrarlar silindikten sonra,615165,137280



EN sınıf dağılımı (temizlik sonrası):
  negative   134587  ✓
  neutral     51719  ✓
  positive   428859  ✓

TR sınıf dağılımı (temizlik sonrası):
  negative    45026  ✓
  neutral     44154  ✓
  positive    48100  ✓


In [ ]:
# 2.4 id ata: filtrelenmiş verideki sıraya göre en_000001, tr_000001 ...
en_temiz["id"] = [f"en_{i:06d}" for i in range(1, len(en_temiz) + 1)]
tr_temiz["id"] = [f"tr_{i:06d}" for i in range(1, len(tr_temiz) + 1)]

# 2.5 Böl: her sınıfı kendi içinde karıştır; ilk 1.000 test, sonraki 1.000 val, sonraki 10.000 train
def bol(df):
    parcalar = {"test": [], "val": [], "train": []}
    for sinif in SINIFLAR:
        s = df[df["label"] == sinif].sample(frac=1, random_state=42)
        parcalar["test"].append(s.iloc[:1000])
        parcalar["val"].append(s.iloc[1000:2000])
        parcalar["train"].append(s.iloc[2000:12000])
    # Üç sınıfı birleştir, dosyayı bir kez daha karıştır, yalnızca id/text/label sütunlarını tut
    return {ad: pd.concat(liste).sample(frac=1, random_state=42)[["id", "text", "label"]].reset_index(drop=True)
            for ad, liste in parcalar.items()}

en_bol = bol(en_temiz)
tr_bol = bol(tr_temiz)

for dil, bolumler in [("EN", en_bol), ("TR", tr_bol)]:
    for ad, df in bolumler.items():
        print(f"{dil} {ad:5s}: {len(df):6d} satır |", df["label"].value_counts().reindex(SINIFLAR).to_dict())
display(tr_bol["test"].head(3))

EN test :   3000 satır | {'negative': 1000, 'neutral': 1000, 'positive': 1000}
EN val  :   3000 satır | {'negative': 1000, 'neutral': 1000, 'positive': 1000}
EN train:  30000 satır | {'negative': 10000, 'neutral': 10000, 'positive': 10000}
TR test :   3000 satır | {'negative': 1000, 'neutral': 1000, 'positive': 1000}
TR val  :   3000 satır | {'negative': 1000, 'neutral': 1000, 'positive': 1000}
TR train:  30000 satır | {'negative': 10000, 'neutral': 10000, 'positive': 10000}


,id,text,label
0,tr_050853,Yorumlar hep olumlu yönde ama ben ne yazıkki s...,neutral
1,tr_125610,Özenle kargolanmış kullanınca yazacağım,neutral
2,tr_039980,tavsiye üzerine almıştım ama kesinlikle çatlak...,neutral


In [ ]:
# 2.6 Kaydet: data/processed/ altına yalnızca id,text,label sütunlarıyla, UTF-8 olarak
OUT_DIR = "data/processed"
os.makedirs(OUT_DIR, exist_ok=True)
for dil, bolumler in [("en", en_bol), ("tr", tr_bol)]:
    for ad, df in bolumler.items():
        df.to_csv(f"{OUT_DIR}/{dil}_{ad}.csv", index=False, encoding="utf-8")

# 2.7 Doğrula: dosyaları diskten yeniden oku ve kuralları kontrol et
BEKLENEN = {"test": 3000, "val": 3000, "train": 30000}
for dil, bolumler in [("en", en_bol), ("tr", tr_bol)]:
    okunan = {}
    for ad in BEKLENEN:
        yol = f"{OUT_DIR}/{dil}_{ad}.csv"
        d = pd.read_csv(yol, encoding="utf-8")
        okunan[ad] = d
        dengeli = (d["label"].value_counts().reindex(SINIFLAR) == BEKLENEN[ad] // 3).all()
        print(f"{yol}: {len(d)} satır (beklenen {BEKLENEN[ad]}) | sütunlar {list(d.columns)} | "
              f"dengeli: {dengeli} | boş hücre: {d.isna().sum().sum()} | "
              f"bellektekiyle aynı: {d.equals(bolumler[ad])} | {os.path.getsize(yol)/1024**2:.1f} MB")
    for a, b in [("test", "val"), ("test", "train"), ("val", "train")]:
        ortak_metin = len(set(okunan[a]["text"]) & set(okunan[b]["text"]))
        ortak_id = len(set(okunan[a]["id"]) & set(okunan[b]["id"]))
        print(f"  {dil}: {a} ∩ {b} → ortak metin {ortak_metin}, ortak id {ortak_id}")

data/processed/en_test.csv: 3000 satır (beklenen 3000) | sütunlar ['id', 'text', 'label'] | dengeli: True | boş hücre: 0 | bellektekiyle aynı: True | 0.5 MB
data/processed/en_val.csv: 3000 satır (beklenen 3000) | sütunlar ['id', 'text', 'label'] | dengeli: True | boş hücre: 0 | bellektekiyle aynı: True | 0.5 MB
data/processed/en_train.csv: 30000 satır (beklenen 30000) | sütunlar ['id', 'text', 'label'] | dengeli: True | boş hücre: 0 | bellektekiyle aynı: True | 5.3 MB
  en: test ∩ val → ortak metin 0, ortak id 0
  en: test ∩ train → ortak metin 0, ortak id 0
  en: val ∩ train → ortak metin 0, ortak id 0
data/processed/tr_test.csv: 3000 satır (beklenen 3000) | sütunlar ['id', 'text', 'label'] | dengeli: True | boş hücre: 0 | bellektekiyle aynı: True | 0.3 MB
data/processed/tr_val.csv: 3000 satır (beklenen 3000) | sütunlar ['id', 'text', 'label'] | dengeli: True | boş hücre: 0 | bellektekiyle aynı: True | 0.3 MB
data/processed/tr_train.csv: 30000 satır (beklenen 30000) | sütunlar ['id', 

In [ ]:
# 2.8 Yalnızca 6 veri dosyasını GitHub'a gönder
!git status --short
!git add data/processed/en_train.csv data/processed/en_val.csv data/processed/en_test.csv \
         data/processed/tr_train.csv data/processed/tr_val.csv data/processed/tr_test.csv
!git commit -m "[Zehra] 6 işlenmiş veri dosyası eklendi (train/val/test, EN ve TR)"
!git pull --rebase
!git push
!git log --oneline -3

?? data/processed/en_test.csv
?? data/processed/en_train.csv
?? data/processed/en_val.csv
?? data/processed/tr_test.csv
?? data/processed/tr_train.csv
?? data/processed/tr_val.csv
[main ea8cbc2] [Zehra] 6 işlenmiş veri dosyası eklendi (train/val/test, EN ve TR)
 6 files changed, 72006 insertions(+)
 create mode 100644 data/processed/en_test.csv
 create mode 100644 data/processed/en_train.csv
 create mode 100644 data/processed/en_val.csv
 create mode 100644 data/processed/tr_test.csv
 create mode 100644 data/processed/tr_train.csv
 create mode 100644 data/processed/tr_val.csv
Current branch main is up to date.
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing objects: 100% (9/9), done.
Writing objects: 100% (10/10), 3.77 MiB | 3.27 MiB/s, done.
Total 10 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/bilgeerol/bim448-sentiment.git